# Renovación de préstamo: flujo refactorizado

Propensión **comercial** a aceptar una renovación. Este notebook organiza el análisis y llama al paquete `loan_renewal`; la lógica está en módulos probados. No determina elegibilidad crediticia.

El original tiene 186 celdas. Se conservan el problema y las familias de modelos, con correcciones metodológicas documentadas en `docs/refactorizacion.md`. Las métricas no son equivalentes a las tablas del original.


## Unidades 1–6 bajo Cookiecutter Data Science

Una sola implementación en `src/loan_renewal`: entorno, contrato raw, DVC, pipeline por etapas, API/Docker y experimentos/Registry. Ver `docs/unidades_01_06.md` y `docs/cookiecutter.md`. El original de `docs` se conserva localmente en `references/original`; este es el único notebook de ejecución versionado.

Instalar desde raíz: `python -m pip install -e '.[dev,api,mlops]'`. Los datos y resultados permanecen privados.


In [ ]:
from pathlib import Path
import sys, json, logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Abre el notebook dentro del repositorio SDG-MLOPS-USE-CASE-LOAN")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))
from loan_renewal.data import load_dataset, validate_dataset, split_temporal
from loan_renewal.schema import FEATURE_COLUMNS, TARGET
from loan_renewal.orchestration import run_stages
from loan_renewal.predict import load_artifact, predict_frame
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
DATA = ROOT / "data/raw/Dataset Renovacion_prestamo.csv"
OUTPUT = ROOT / "models"
REPORTS = ROOT / "reports/training"
if not DATA.exists():
    raise FileNotFoundError(f"Coloca el CSV original en {DATA}")


## 1. Ingestión y contrato

CSV separado por `;`, etiqueta `FLAG_VENTA=1`. `CLIENTE` y `MES` sirven para validación y partición, nunca como predictores. El diccionario llama `USO_TRIM_LINEA_Bco` a la variable `USO_TRIM_LINEA_BBVA` del CSV. La presentación menciona dos meses; el CSV contiene nueve.


In [ ]:
df = load_dataset(DATA)
profile = validate_dataset(df)
display(pd.DataFrame({"conteo": df[TARGET].value_counts().sort_index(), "proporcion": df[TARGET].value_counts(normalize=True).sort_index()}))
display(pd.DataFrame({"nulos": df.isna().sum(), "porcentaje": df.isna().mean().mul(100)}).query("nulos > 0"))
print({k: profile[k] for k in ["rows", "columns", "unique_customers", "months"]})


## 2. Partición antes de aprender transformaciones

Entrenamiento: enero–junio de 2015. Validación: julio–agosto. Test reservado: septiembre. La imputación, escalado, vocabulario categórico y KMeans se ajustan exclusivamente en train. El test se usa para reportar el candidato seleccionado, sin reentrenar con validación.


In [ ]:
parts = split_temporal(df)
display(pd.DataFrame([{"particion": name, "filas": len(frame), "positivos": int(frame[TARGET].sum()), "meses": sorted(frame.MES.unique().tolist())} for name, frame in parts.items()]))


## 3. Exploración de entrenamiento

Los gráficos se calculan sobre train. El análisis histórico del original ya exploró el archivo completo: este corte corrige el nuevo procedimiento, pero una campaña futura será necesaria para una evaluación prospectiva nueva.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
parts["train"][TARGET].value_counts().sort_index().plot.bar(ax=axes[0], color=["#64748b", "#f97316"], title="Distribución de renovación en train")
axes[0].set_xlabel("FLAG_VENTA")
axes[0].set_ylabel("Registros")
np.log1p(parts["train"]["LINEA_RENOVADO"].clip(lower=0)).hist(ax=axes[1], bins=35, color="#2563eb")
axes[1].set_title("Monto ofrecido en train, log1p")
axes[1].set_xlabel("log1p(monto)")
fig.tight_layout()
plt.show()


## 4. Pipeline, selección y MLflow

Se comparan seis candidatos base y GridSearchCV temporal (4 combinaciones × 3 folds). Transformaciones y balanceo se aprenden dentro de cada fold. Modelo y umbral se seleccionan por F1 en validación con AP como desempate; septiembre se reporta al final.

`run_stages` genera evidencia de ingestión, contrato, entrenamiento, evaluación y tracking. Guarda el bundle en `models` y métricas en `reports/training`; MLflow utiliza SQLite en `mlruns`. Repetir esta comprobación técnica no crea un nuevo test independiente. No ajustar candidatos a partir de septiembre.


In [ ]:
ENABLE_TUNING = True
stages = run_stages(DATA, OUTPUT, tune=ENABLE_TUNING, track=True, report_dir=REPORTS)
assert stages["status"] == "ok"
report = json.loads((REPORTS / "metrics.json").read_text())
tracking = json.loads((REPORTS / "tracking.json").read_text())
assert tracking["reload_prediction_parity"]
display(pd.DataFrame(stages["stages"])[["stage", "status", "seconds"]])
comparison = pd.read_csv(REPORTS / "comparison.csv")
display(comparison[["candidate", "validation_precision", "validation_recall", "validation_f1", "validation_average_precision"]])
print("Modelo:", report["winner"], "umbral:", report["threshold"], "MLflow run:", tracking["run_id"])


## 5. Evaluación final del seleccionado

ROC-AUC y Average Precision usan scores, no etiquetas duras. La precisión positiva, recall y F1 se calculan con el umbral elegido en validación. La accuracy se interpreta junto al desbalance. No se copian resultados del notebook anterior.


In [ ]:
display(pd.DataFrame({"validation": report["validation"], "test": report["test"]}).drop(index=["confusion_matrix"]))
display(pd.DataFrame(report["test"]["confusion_matrix"], index=["real 0", "real 1"], columns=["predicho 0", "predicho 1"]))
print("Paridad tras recarga:", report["reload_prediction_parity"])


## 6. Inferencia desde variables originales

El artefacto contiene transformaciones, modelo y umbral. No construir columnas `_LOG`, one-hot o clusters manualmente. Se rechazan features ausentes, extras y valores infinitos. Las categorías desconocidas se admiten con el vocabulario aprendido en train.


In [ ]:
bundle = load_artifact(OUTPUT / "model.joblib")
# Comprobación técnica de paridad con entradas del holdout; no es una recomendación de contacto.
raw_examples = parts["test"].loc[:, FEATURE_COLUMNS].head(5)
predictions = predict_frame(bundle, raw_examples)
assert len(predictions) == len(raw_examples)
assert np.isfinite(predictions["probabilidad_renovacion"]).all()
print("Contrato raw validado; inferencia finita para", len(predictions), "registros.")
print("Hash de datos:", report["manifest"]["dataset_sha256"])


## 7. API y paridad

La API reutiliza el artefacto completo. La prueba HTTP usa entradas sintéticas; verifica el transporte, no rendimiento predictivo. Docker se comprueba por separado.


In [ ]:
from fastapi.testclient import TestClient
from api.app import create_app
from loan_renewal.tracking import synthetic_probe
probe = synthetic_probe()
offline = predict_frame(bundle, probe)
with TestClient(create_app(OUTPUT / "model.joblib")) as client:
    assert client.get("/health").status_code == 200
    for i, row in enumerate(probe.to_dict(orient="records")):
        response = client.post("/predecir", json=row)
        assert response.status_code == 200
        np.testing.assert_allclose(response.json()["score_propension"], offline.iloc[i]["probabilidad_renovacion"], rtol=1e-12, atol=1e-12)
print("API: readiness y paridad offline/HTTP verificadas con entradas sintéticas.")


## 8. DVC, Docker, Registry y límites

Desde raíz y con `.venv` activa:

```bash
python -m loan_renewal version-data --dvc
# dvc repro usa el mismo pipeline; no publica datos ni configura remotos.
dvc repro
docker compose up --build -d
python scripts/check_service.py
python -m loan_renewal experiments all
```

`experiments all` registra base/grid/random y selecciona por validación antes de asignar el alias `candidato`; no despliega ni aprueba producción. Cada experimento repite el reporte histórico de septiembre: no utilizarlo para seleccionar. Los scores no están calibrados. Faltan costes, validación de disponibilidad de variables y evaluación prospectiva.

El notebook no inicia Docker ni publica datos automáticamente. Estas operaciones y el notebook son evidencias distintas. Ver `docs/refactorizacion.md`, `docs/diccionario.md` y `docs/cookiecutter.md`.
